<a href="https://colab.research.google.com/github/DhimanTarafdar/AAA/blob/main/Cat_vs_Dog_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import Libraries**

In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import kagglehub
import matplotlib.pyplot as plt

# **Download Dataset**

In [5]:
path = kagglehub.dataset_download("salader/dogsvscats")
print("path", path)

Using Colab cache for faster access to the 'dogsvscats' dataset.
path /kaggle/input/dogsvscats


# **Setup**

In [6]:
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

Using Device: cuda


# **Path Setup**

In [7]:
TRAIN_PATH = os.path.join(path, "train")
VAL_PATH   = os.path.join(path, "test")

print("Train Path:", TRAIN_PATH)
print("Val Path  :", VAL_PATH)

Train Path: /kaggle/input/dogsvscats/train
Val Path  : /kaggle/input/dogsvscats/test


# **Image Transformation**

In [8]:
from torchvision import transforms

# ✅ Train Transform (with augmentation)
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),

    transforms.RandomHorizontalFlip(p=0.5),   # image flip
    transforms.RandomRotation(10),            # rotate
    transforms.RandomResizedCrop(128, scale=(0.8, 1.0)),  # zoom effect

    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])


# ✅ Validation/Test Transform (NO augmentation)
val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

# **Custom Dataset**

In [9]:
class BinaryClassification(Dataset):
    def __init__(self, root_dir, transform=None):
        super().__init__()

        self.samples = []
        self.transform = transform

        self.classes = sorted([d for d in os.listdir(root_dir)
                               if os.path.isdir(os.path.join(root_dir, d))])

        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}

        for class_name in self.classes:
            class_path = os.path.join(root_dir, class_name)

            for img_name in os.listdir(class_path):
                img_path = os.path.join(class_path, img_name)

                if os.path.isfile(img_path):
                    label = self.class_to_idx[class_name]
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# **Load Data**

In [10]:
from torch.utils.data import random_split

# Full dataset (without augmentation first)
full_dataset = BinaryClassification(TRAIN_PATH, transform=val_transform)

# Split
train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Apply train transform manually
train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform   = val_transform

# **Subset + DataLoader**

In [11]:
train_dataset = train_dataset
val_dataset   = val_dataset

In [12]:
pin = True if device.type == 'cuda' else False
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  pin_memory=pin)
test_loader  = DataLoader(val_dataset,  batch_size=32, shuffle=False, pin_memory=pin)

# **CNN Architecture**

In [13]:
class CatDogCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 128),  # reduced params
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# **Define Model**

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CatDogCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.0003, weight_decay=1e-4)

# **Model Summary**

In [15]:
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters    : {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters    : 359,970
Trainable Parameters: 359,970


# **Training Setup**

In [16]:
learning_rate = 0.0003
epochs        = 15

# **Training Loop**

In [17]:
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels   = batch_labels.to(device)

        outputs = model(batch_features)
        loss    = criterion(outputs, batch_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/15, Loss: 0.6464
Epoch 2/15, Loss: 0.5481
Epoch 3/15, Loss: 0.5073
Epoch 4/15, Loss: 0.4763
Epoch 5/15, Loss: 0.4442
Epoch 6/15, Loss: 0.4157
Epoch 7/15, Loss: 0.3925
Epoch 8/15, Loss: 0.3697
Epoch 9/15, Loss: 0.3497
Epoch 10/15, Loss: 0.3304
Epoch 11/15, Loss: 0.3166
Epoch 12/15, Loss: 0.2937
Epoch 13/15, Loss: 0.2772
Epoch 14/15, Loss: 0.2605
Epoch 15/15, Loss: 0.2467


# **Evaluation Function**

In [20]:
def evaluate(loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item()

            # Accuracy calculation
            _, predicted = torch.max(out, 1)
            correct += (predicted == y).sum().item()
            total += y.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total

    return avg_loss, accuracy

In [22]:
train_loss, train_acc = evaluate(train_loader)
val_loss, val_acc     = evaluate(test_loader)

print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.4f}")

Train Loss: 0.2008, Train Acc: 0.9167
Val   Loss: 0.3139, Val   Acc: 0.8620


# **Prediction Function for Unseen Images**

In [ ]:
def predict_image(image_path):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(device) # Batch dimension add করা

    with torch.no_grad():
        outputs = model(image)
        _, predicted = torch.max(outputs, 1)

    # (0=Cat, 1=Dog)
    class_names = ['Cat', 'Dog']
    result = class_names[predicted.item()]


    plt.imshow(Image.open(image_path))
    plt.title(f"Prediction: {result}")
    plt.axis('off')
    plt.show()

# **Test with your uploaded image**
# Colab-এ ছবি আপলোড করে তার পাথ এখানে দাও
my_image_path = "cat_or_dog.jpg"
predict_image(my_image_path)